# Task 2: Automated Data Preprocessing Pipeline

**Objective:** Build a reusable Scikit-learn preprocessing pipeline that handles missing numerical and categorical values, applies One-Hot Encoding to categorical columns and MinMax scaling to numerical columns.

**Dataset:** `customer_data.csv` from Task 1.

**Key design:** use `SimpleImputer` + `MinMaxScaler` for numerical features and `SimpleImputer` + `OneHotEncoder` for categorical features, combined using `ColumnTransformer` and `Pipeline`.

In [ ]:
# Install if required
# %pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

raw = pd.read_csv("customer_data.csv")
raw.head()

## 1. Separate features and exclude identifier

In [ ]:
# customer_id is an identifier, not a predictive feature
X = raw.drop(columns=["customer_id"])

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("\\nMissing values before preprocessing:")
print(X.isna().sum())

## 2. Build numerical and categorical pipelines

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

full_pipeline = Pipeline(steps=[("preprocessor", preprocessor)])

## 3. Fit and transform the data

In [ ]:
processed_array = full_pipeline.fit_transform(X)

feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()
processed = pd.DataFrame(processed_array, columns=feature_names, index=X.index)

print("Original feature shape:", X.shape)
print("Processed feature shape:", processed.shape)
print("\\nFirst five processed rows:")
display(processed.head())

## 4. Validate preprocessing

In [ ]:
print("Any missing values remaining?", processed.isna().any().any())
print("\\nMinimum values in scaled numeric features:")
print(processed[[f"num__{c}" for c in numeric_features]].min())
print("\\nMaximum values in scaled numeric features:")
print(processed[[f"num__{c}" for c in numeric_features]].max())

## 5. Save processed output

In [ ]:
processed.to_csv("customer_data_preprocessed.csv", index=False)
print("Saved: customer_data_preprocessed.csv")

## 6. Conclusion

The preprocessing pipeline performs the required operations in a repeatable sequence:
1. Numerical missing values are filled using the median.
2. Categorical missing values are filled using the most frequent category.
3. Categorical columns are converted into binary indicator columns using One-Hot Encoding.
4. Numerical columns are scaled to the 0–1 range using MinMaxScaler.

**Important:** For a predictive ML project, split training and test data first, then fit the pipeline only on training data. This prevents data leakage. This task demonstrates preprocessing; it does not train a prediction model.